<a href="https://colab.research.google.com/github/weagan/In-Context-Learning/blob/main/implicit_icrl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implicit ICRL Demo

Demonstrates learning via context (no weight updates).


In [1]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.0 MB/s eta 0:00:00


In [2]:
import os
from groq import Groq
from google.colab import userdata

# Get API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)
MODEL = 'llama-3.3-70b-versatile' # Updated to a currently supported smaller model

In [3]:
for model_obj in client.models.list().data:
    print(model_obj.id)

openai/gpt-oss-20b
moonshotai/kimi-k2-instruct
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
groq/compound
whisper-large-v3
llama-3.3-70b-versatile
llama-3.1-8b-instant
qwen/qwen3-32b
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
moonshotai/kimi-k2-instruct-0905
groq/compound-mini
openai/gpt-oss-safeguard-20b
meta-llama/llama-4-scout-17b-16e-instruct
whisper-large-v3-turbo


In [4]:
def call_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    return response.choices[0].message.content

In [5]:
def ask(problem, context=''):
    prompt = context + '\nSolve: ' + problem + '\nAnswer:'
    return call_llm(prompt)

In [6]:
# Baseline
print(ask("Translate 'The sun is blue' into Keldorian."))

To translate "The sun is blue" into Keldorian, I'll need to create a fictional language for Keldorian. Since there's no established language for Keldorian, I'll generate a translation based on a hypothetical language structure.

In Keldorian, let's assume the language has a subject-verb-object word order, similar to English. Here's a possible translation:

"Zhilak aethon niamon"

Breakdown:
- "Zhilak" means "sun" in Keldorian.
- "Aethon" is the verb "to be" in the present tense, which can be translated to "is" in English.
- "Niamon" means "blue" in Keldorian.

So, the translation of "The sun is blue" into Keldorian would be "Zhilak aethon niamon".


In [7]:
# Good examples (implicit reward)
good_context = '''
Solve: English: The cat is red. -> Keldorian: Boras zin kalar.
English: The dog is big. -> Keldorian: Garm zin mular.
English: The water is cold. -> Keldorian: Aqua zin bular.
'''

print(ask('English: The sun is blue. -> Keldorian:', good_context))

To solve this, we need to identify the pattern in the given translations from English to Keldorian.

1. English: The cat is red. -> Keldorian: Boras zin kalar.
2. English: The dog is big. -> Keldorian: Garm zin mular.
3. English: The water is cold. -> Keldorian: Aqua zin bular.

From these examples, we can observe the following patterns:
- "The" is not directly translated but seems to be part of the sentence structure.
- The subject (cat, dog, water) is translated to its Keldorian equivalent (Boras, Garm, Aqua).
- "is" is translated to "zin".
- The adjective (red, big, cold) is translated to its Keldorian equivalent (kalar, mular, bular).

Given the pattern, we need to translate "The sun is blue" into Keldorian. Assuming "sun" and "blue" follow the pattern:
- "The" is part of the structure.
- "sun" needs its Keldorian equivalent, which is not provided directly but following the pattern of direct translation for subjects, let's assume it's "Sol" (since "Aqua" for water, "Boras" for cat,

In [8]:
# bad examples (implicit reward)
# Attempting a consistently incorrect pattern to show difference
bad_context = '''
Solve: Count the number of primes between 1 and 5.
Answer: 0

Solve: Count the number of primes between 1 and 7.
Answer: 0
'''

print(ask('Count the number of primes between 1 and 10.', bad_context))

To solve this, we need to identify the prime numbers between 1 and 10. 

Prime numbers are numbers greater than 1 that have no divisors other than 1 and themselves. 

The numbers between 1 and 10 are: 2, 3, 4, 5, 6, 7, 8, 9, 10.

From these, the prime numbers are: 2, 3, 5, 7.

So, the number of primes between 1 and 10 is 4. 

Answer: 4.


### New Example: Text Transformation (Reverse and Capitalize)

This example demonstrates how implicit context can guide the model to produce output in a very specific format, even for tasks it can perform without context. We'll ask the model to reverse words in a phrase and capitalize the first letter of each reversed word.

#### No Context Example

In [9]:
print(ask("Reverse the words and capitalize the first letter of each word in the phrase 'Artificial Intelligence'."))

The phrase 'Artificial Intelligence' spelled backwards is 'Ecneligettis Laicifitra'. 

Capitalizing the first letter of each word in the reversed phrase: 'Ecneligettis' and 'Laicifitra' are the reversed words. The capitalized version is: 'Ecneligettis Laicifitra'.


#### Good Examples (Implicit Context)

Here, we provide examples of the desired transformation. Notice how the model is expected to learn the specific output format (e.g., `Original Phrase -> Transformed Phrase`) and apply it, along with the transformation logic, to the new input.

In [11]:
transform_context = '''
Solve: Reverse and capitalize: Hello World -> dlroW olleH
Solve: Reverse and capitalize: Python Code -> edoC nohtyP
'''

print(ask('Reverse and capitalize: Large Language Model ->', transform_context))
print(ask('Reverse and capitalize: Artificial Intelligence ->', transform_context))

To solve this, we need to reverse the order of the words and then capitalize them. 

The original phrase is: Large Language Model

Reversing the order of the words: Model Language Large

Reversing each word:
- Model -> Ledom
- Language -> Egaugnal
- Large -> Egarl

Combining the reversed words: Ledom Egaugnal Egarl

So, the answer is: Ledom Egaugnal Egarl
To solve this, we need to reverse the order of the words and then capitalize them. 

The original phrase is: Artificial Intelligence

Reversing the order of the words: Intelligence Artificial

Reversing the letters in each word: 
- Intelligence -> ecnegilletni
- Artificial -> lacitretfa

Combining the reversed words: ecnegilletni lacitretfa

So, the answer is: ecnegilletni lacitretfa


isnt this the same as multi shot context prompting

Yes, you're absolutely right! In the context of this notebook, 'implicit reward' is essentially achieved through what's known as **multi-shot context prompting** (also commonly called few-shot prompting).

**Multi-shot context prompting** is the technique of providing several examples (the 'shots') within your prompt to an LLM. These examples serve as a guide for the model, showing it the desired input-output format, style, or behavior for a new, similar query.

The 'implicit reward' aspect comes from the _effect_ of this technique: the model is implicitly 'rewarded' or guided to produce an output that adheres to the patterns and structure it observed in those provided examples. It learns what is expected and attempts to replicate that behavior without any explicit feedback loop or model weight updates. So, you can think of implicit reward as the successful outcome of multi-shot prompting when the model successfully mimics the desired behavior shown in the examples.